# 3D object plotly

In [3]:
import plotly
plotly.__version__
import numpy as np
import meshio  
import plotly.graph_objects as go

In [4]:
mesh_data = meshio.read('Images/20240304uitkijktoren.ply')
vertices = mesh_data.points
triangles =  mesh_data.cells['triangle']

Expected `element vertex` or `element face` or `obj_info`, got `element material 1`


Error: Couldn't read file Images\20240304uitkijktoren.ply as ply

SystemExit: 1

In [ ]:
from numpy import sin, cos, pi
def rot_x(t):
    return np.array([[1, 0, 0],
                    [0, cos(t), -sin(t)],
                    [0, sin(t), cos(t)]])
def rot_z(t):
    return np.array([[cos(t), -sin(t), 0],
                    [sin(t), cos(t), 0],
                    [0, 0, 1]])

In [ ]:
#apply rot_z(angle2) * rot_x(angle1)
A = rot_x(pi/4)
B = rot_z(4*pi/9+pi/4)

#Apply the product of the two rotations to the object vertices:
new_vertices = np.einsum('ik, kj -> ij',  vertices, (np.dot(B, A)).T)#new_vertices has the shape (n_vertices, 3)

In [ ]:
x, y, z = new_vertices.T
I, J, K = triangles.T

tri_points = new_vertices[triangles] 

In [ ]:
pl_mygrey=[0, 'rgb(153, 153, 153)'], [1., 'rgb(255,255,255)']
                           
pl_mesh = go.Mesh3d(x=x,
                    y=y,
                    z=z,
                    colorscale=pl_mygrey, 
                    intensity= z,
                    flatshading=True,
                    i=I,
                    j=J,
                    k=K,
                    name='Beethoven',
                    showscale=False
                    )

In [ ]:
pl_mesh.update(cmin=-7,# atrick to get a nice plot (z.min()=-3.31909)
               lighting=dict(ambient=0.18,
                             diffuse=1,
                             fresnel=0.1,
                             specular=1,
                             roughness=0.05,
                             facenormalsepsilon=1e-15,
                             vertexnormalsepsilon=1e-15),
               lightposition=dict(x=100,
                                  y=200,
                                  z=0
                                 )
                      );

In [ ]:
Xe = []
Ye = []
Ze = []
for T in tri_points:
    Xe.extend([T[k%3][0] for k in range(4)]+[ None])
    Ye.extend([T[k%3][1] for k in range(4)]+[ None])
    Ze.extend([T[k%3][2] for k in range(4)]+[ None])
       
#define the trace for triangle sides
lines = go.Scatter3d(
                   x=Xe,
                   y=Ye,
                   z=Ze,
                   mode='lines',
                   name='',
                   line=dict(color= 'rgb(70,70,70)', width=1))

In [ ]:
layout = go.Layout(
         title="Beethoven<br>Mesh3d with flatshading",
         font=dict(size=16, color='white'),
         width=700,
         height=700,
         scene_xaxis_visible=False,
         scene_yaxis_visible=False,
         scene_zaxis_visible=False,
         paper_bgcolor='rgb(50,50,50)',
       
        )

In [ ]:
fig = go.Figure(data=[pl_mesh, lines], layout=layout)

In [ ]:
import chart_studio.plotly as py

py.iplot(fig, filename='mesh3d-flatshading'